# Global Retail data modelling

Create different schemas for differnt layers
Initialize Bronze, Silver, and Gold schemas.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

# 1. Starting with the Bronze layer, load the raw data into a Delta table.

Read all six source tables from the CRM and ERP systems. 

In [0]:
crm_cust_info=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/cust_info.csv")

In [0]:
crm_prd_info=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/prd_info.csv")

In [0]:
crm_sales_details=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/sales_details.csv")

In [0]:
erp_Cust_az12=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/CUST_AZ12.csv")

In [0]:
erp_Loc_a101=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/LOC_A101.csv")

In [0]:
erp_Px_cat_g1v2=spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv("/Workspace/Users/kaurkamal98700@gmail.com/PX_CAT_G1V2.csv")

# Save the DataFrame to the Bronze layer in Delta format.

In [0]:
crm_cust_info.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.crm_cust_info")

In [0]:
crm_prd_info.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.crm_prd_info")

In [0]:
crm_sales_details.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.crm_sales_details")

In [0]:
erp_Cust_az12.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.erp_Cust_az12")

In [0]:
erp_Loc_a101.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.erp_Loc_a101")

In [0]:
erp_Px_cat_g1v2.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.erp_Px_cat_g1v2")

# Validation

In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,crm_cust_info,false
bronze,crm_prd_info,false
bronze,crm_sales_details,false
bronze,erp_cust_az12,false
bronze,erp_loc_a101,false
bronze,erp_px_cat_g1v2,false


# 2. Transform and clean the raw data in the Silver layer.

In [0]:
# # Load the CRM customer data from the Bronze layer for transformation.
crm_cust_info_df = spark.table("bronze.crm_cust_info")

In [0]:
crm_cust_info_df.printSchema()

root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: date (nullable = true)



# Data Check/ Validation

In [0]:
# Checking null or missing values
from pyspark.sql.functions import col, sum, when

crm_cust_info_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crm_cust_info_df.columns
])

DataFrame[cst_id: bigint, cst_key: bigint, cst_firstname: bigint, cst_lastname: bigint, cst_marital_status: bigint, cst_gndr: bigint, cst_create_date: bigint]

In [0]:
# checking duplicates
crm_cust_info_df.count()

crm_cust_info_df.dropDuplicates().count()

18494

In [0]:
# Check for duplicate customer IDs in the CRM customer information.
from pyspark.sql.functions import count

crm_cust_info_df.groupBy("cst_id") \
           .count() \
           .filter("count > 1") 

DataFrame[cst_id: int, count: bigint]

In [0]:
# Display distinct customer gender values for data validation.
crm_cust_info_df.select("cst_gndr").distinct()

DataFrame[cst_gndr: string]

In [0]:
# # Display distinct customer marital status for data validation.
crm_cust_info_df.select("cst_marital_status").distinct()

DataFrame[cst_marital_status: string]

# Data Cleaning

In [0]:
# Filter out invalid CRM customer records with null key fields.
from pyspark.sql import functions as F

crm_cust_info_df = crm_cust_info_df.filter(
    F.col("cst_id").isNotNull() &
    F.col("cst_create_date").isNotNull()
)

In [0]:
crm_cust_info_df = crm_cust_info_df.fillna({
    "cst_firstname": "Unknown",
    "cst_lastname": "Unknown",
    "cst_marital_status": "Unknown",
    "cst_gndr": "Unknown"
})

In [0]:
# Trim whitespace from customer string fields to standardize the data.
crm_cust_info_df = (
    crm_cust_info_df
    .withColumn("cst_firstname", F.trim(F.col("cst_firstname")))
    .withColumn("cst_lastname", F.trim(F.col("cst_lastname")))
    .withColumn("cst_marital_status", F.trim(F.col("cst_marital_status")))
    .withColumn("cst_gndr", F.trim(F.col("cst_gndr")))
)

In [0]:
# Standardizing data for gender
crm_cust_info_df = crm_cust_info_df.withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
     .when(F.upper(F.col("cst_gndr")) == "F", "Female")
     .otherwise("Unknown")
)

In [0]:
# Standardizing data for marital_status
crm_cust_info_df = crm_cust_info_df.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
     .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
     .otherwise("Unknown")
)

In [0]:
# Remove duplicate customer records by keeping the latest record based on creation date.
from pyspark.sql.window import Window

window_spec = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

crm_cust_info_df = (
    crm_cust_info_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Saving as delta table in silver schema

In [0]:
crm_cust_info_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_cust_info")

# Doing almost same procedure of data read-> data check/validate->data cleaning-> saving as delta table
# for all 6 tables

In [0]:
erp_cust_az12_df = spark.table("bronze.erp_cust_az12")

In [0]:
erp_cust_az12_df.printSchema()

root
 |-- CID: string (nullable = true)
 |-- BDATE: date (nullable = true)
 |-- GEN: string (nullable = true)



In [0]:
# Missing values
from pyspark.sql.functions import col, sum, when

erp_cust_az12_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in erp_cust_az12_df.columns
])

DataFrame[CID: bigint, BDATE: bigint, GEN: bigint]

In [0]:
# Duplicate rows
erp_cust_az12_df.count()

erp_cust_az12_df.dropDuplicates().count()

18484

In [0]:
# Duplicate Customer IDs
from pyspark.sql.functions import count

erp_cust_az12_df.groupBy("CID") \
                .count() \
                .filter("count > 1")

DataFrame[CID: string, count: bigint]

In [0]:
# Distinct Gender values
erp_cust_az12_df.select("GEN").distinct()

DataFrame[GEN: string]

In [0]:
erp_cust_az12_df = (
    erp_cust_az12_df
    .withColumn("CID", F.trim(F.col("CID")))
    .withColumn("GEN", F.trim(F.col("GEN")))
)

In [0]:
erp_cust_az12_df = erp_cust_az12_df.withColumn(
    "GEN",
    F.when(
        (F.col("GEN").isNull()) | (F.col("GEN") == ""), "Unknown"
    )
    .when(F.upper(F.col("GEN")) == "M", "Male")
    .when(F.upper(F.col("GEN")) == "MALE", "Male")
    .when(F.upper(F.col("GEN")) == "F", "Female")
    .when(F.upper(F.col("GEN")) == "FEMALE", "Female")
    .otherwise("Unknown")
)

In [0]:
erp_cust_az12_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_cust_az12")

In [0]:
erp_loc_a101_df = spark.table("bronze.erp_loc_a101")

In [0]:
erp_loc_a101_df.printSchema()

root
 |-- CID: string (nullable = true)
 |-- CNTRY: string (nullable = true)



In [0]:
# Missing values
from pyspark.sql.functions import col, sum, when

erp_loc_a101_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in erp_loc_a101_df.columns
])

DataFrame[CID: bigint, CNTRY: bigint]

In [0]:
# Duplicate Customer IDs
from pyspark.sql.functions import count

erp_loc_a101_df.groupBy("CID") \
               .count() \
               .filter("count > 1")

DataFrame[CID: string, count: bigint]

In [0]:
# Distinct Country Values
erp_loc_a101_df.select("CNTRY").distinct()

DataFrame[CNTRY: string]

In [0]:
erp_loc_a101_df = (
    erp_loc_a101_df
    .withColumn("CID", F.trim(F.col("CID")))
    .withColumn("CNTRY", F.trim(F.col("CNTRY")))
)

In [0]:
# standardizing data for country column
erp_loc_a101_df = erp_loc_a101_df.withColumn(
    "CNTRY",
    F.when(
        (F.col("CNTRY").isNull()) | (F.col("CNTRY") == ""),
        "Unknown"
    )
    .when(F.upper(F.col("CNTRY")) == "US", "United States")
    .when(F.upper(F.col("CNTRY")) == "USA", "United States")
    .when(F.upper(F.col("CNTRY")) == "UNITED STATES", "United States")
    .when(F.upper(F.col("CNTRY")) == "DE", "Germany")
    .when(F.upper(F.col("CNTRY")) == "GERMANY", "Germany")
    .otherwise(F.col("CNTRY"))
)

In [0]:
erp_loc_a101_df = erp_loc_a101_df.dropDuplicates()

In [0]:
erp_loc_a101_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_loc_a101")

In [0]:
crm_prd_info_df = spark.table("bronze.crm_prd_info")

In [0]:
crm_prd_info_df.printSchema()

root
 |-- prd_id: integer (nullable = true)
 |-- prd_key: string (nullable = true)
 |-- prd_nm: string (nullable = true)
 |-- prd_cost: integer (nullable = true)
 |-- prd_line: string (nullable = true)
 |-- prd_start_dt: date (nullable = true)
 |-- prd_end_dt: date (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum, when

crm_prd_info_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crm_prd_info_df.columns
])

DataFrame[prd_id: bigint, prd_key: bigint, prd_nm: bigint, prd_cost: bigint, prd_line: bigint, prd_start_dt: bigint, prd_end_dt: bigint]

In [0]:
from pyspark.sql.functions import count

crm_prd_info_df.groupBy("prd_id") \
               .count() \
               .filter("count > 1") \
               .show()

+------+-----+
|prd_id|count|
+------+-----+
+------+-----+



In [0]:
crm_prd_info_df.select("prd_line").distinct().show(truncate=False)

+--------+
|prd_line|
+--------+
|R       |
|S       |
|M       |
|T       |
|NULL    |
+--------+



In [0]:
crm_prd_info_df.filter("prd_cost < 0").show()

+------+-------+------+--------+--------+------------+----------+
|prd_id|prd_key|prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt|
+------+-------+------+--------+--------+------------+----------+
+------+-------+------+--------+--------+------------+----------+



In [0]:
# # Check for invalid product records where the end date is before the start date.
crm_prd_info_df.filter("prd_end_dt < prd_start_dt")


DataFrame[prd_id: int, prd_key: string, prd_nm: string, prd_cost: int, prd_line: string, prd_start_dt: date, prd_end_dt: date]

In [0]:
crm_prd_info_df = crm_prd_info_df.fillna({
    "prd_cost": 0,
    "prd_line": "Unknown"
})

In [0]:
crm_prd_info_df = (
    crm_prd_info_df
    .withColumn("prd_key", F.trim("prd_key"))
    .withColumn("prd_nm", F.trim("prd_nm"))
    .withColumn("prd_line", F.trim("prd_line"))
)

In [0]:
crm_prd_info_df = crm_prd_info_df.withColumn(
    "prd_line",
    F.when(F.col("prd_line")=="M","Mountain")
     .when(F.col("prd_line")=="R","Road")
     .when(F.col("prd_line")=="S","Accessories")
     .when(F.col("prd_line")=="T","Touring")
     .otherwise("Unknown")
)

In [0]:
# Calculate the next product start date to support product record validation.
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

crm_prd_info_df = crm_prd_info_df.withColumn(
    "next_start_date",
    F.lead("prd_start_dt").over(window_spec)
)

In [0]:
# Update product end dates using the next product start date to maintain valid date ranges.
crm_prd_info_df = crm_prd_info_df.withColumn(
    "prd_end_dt",
    F.when(
        F.col("next_start_date").isNull(),
        F.lit(None).cast("date")
    ).otherwise(F.date_sub(F.col("next_start_date"),1))
).drop("next_start_date")

In [0]:
crm_prd_info_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_prd_info")

In [0]:
erp_px_cat_g1v2_df = spark.table("bronze.erp_Px_cat_g1v2")

In [0]:
erp_px_cat_g1v2_df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- CAT: string (nullable = true)
 |-- SUBCAT: string (nullable = true)
 |-- MAINTENANCE: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum, when

erp_px_cat_g1v2_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in erp_px_cat_g1v2_df.columns
]).show()

+---+---+------+-----------+
| ID|CAT|SUBCAT|MAINTENANCE|
+---+---+------+-----------+
|  0|  0|     0|          0|
+---+---+------+-----------+



In [0]:
from pyspark.sql.functions import count

erp_px_cat_g1v2_df.groupBy("ID") \
                  .count() \
                  .filter("count > 1") \
                  .show()

+---+-----+
| ID|count|
+---+-----+
+---+-----+



In [0]:
erp_px_cat_g1v2_df.select("CAT").distinct().show(truncate=False)

+-----------+
|CAT        |
+-----------+
|Accessories|
|Bikes      |
|Clothing   |
|Components |
+-----------+



In [0]:
erp_px_cat_g1v2_df.select("SUBCAT").distinct().show(truncate=False)

+-----------------+
|SUBCAT           |
+-----------------+
|Bike Racks       |
|Bike Stands      |
|Bottles and Cages|
|Cleaners         |
|Fenders          |
|Helmets          |
|Hydration Packs  |
|Lights           |
|Locks            |
|Panniers         |
|Pumps            |
|Tires and Tubes  |
|Mountain Bikes   |
|Road Bikes       |
|Touring Bikes    |
|Bib-Shorts       |
|Caps             |
|Gloves           |
|Jerseys          |
|Shorts           |
+-----------------+
only showing top 20 rows


In [0]:
erp_px_cat_g1v2_df.select("MAINTENANCE").distinct().show(truncate=False)

+-----------+
|MAINTENANCE|
+-----------+
|Yes        |
|No         |
+-----------+



In [0]:
erp_px_cat_g1v2_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_px_cat_g1v2")

In [0]:
crm_sales_details_df = spark.table("bronze.crm_sales_details")

In [0]:
crm_sales_details_df.printSchema()

root
 |-- sls_ord_num: string (nullable = true)
 |-- sls_prd_key: string (nullable = true)
 |-- sls_cust_id: integer (nullable = true)
 |-- sls_order_dt: integer (nullable = true)
 |-- sls_ship_dt: integer (nullable = true)
 |-- sls_due_dt: integer (nullable = true)
 |-- sls_sales: integer (nullable = true)
 |-- sls_quantity: integer (nullable = true)
 |-- sls_price: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum, when

crm_sales_details_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crm_sales_details_df.columns
])

DataFrame[sls_ord_num: bigint, sls_prd_key: bigint, sls_cust_id: bigint, sls_order_dt: bigint, sls_ship_dt: bigint, sls_due_dt: bigint, sls_sales: bigint, sls_quantity: bigint, sls_price: bigint]

In [0]:
crm_sales_details_df.count()



60398

In [0]:
crm_sales_details_df.dropDuplicates().count()

60398

In [0]:
# the date format is not correct here
crm_sales_details_df.select(
    "sls_order_dt",
    "sls_ship_dt",
    "sls_due_dt"
).show(20, False)

+------------+-----------+----------+
|sls_order_dt|sls_ship_dt|sls_due_dt|
+------------+-----------+----------+
|20101229    |20110105   |20110110  |
|20101229    |20110105   |20110110  |
|20101229    |20110105   |20110110  |
|20101229    |20110105   |20110110  |
|20101229    |20110105   |20110110  |
|20101230    |20110106   |20110111  |
|20101230    |20110106   |20110111  |
|20101230    |20110106   |20110111  |
|20101230    |20110106   |20110111  |
|20101231    |20110107   |20110112  |
|20101231    |20110107   |20110112  |
|20101231    |20110107   |20110112  |
|20101231    |20110107   |20110112  |
|20101231    |20110107   |20110112  |
|20110101    |20110108   |20110113  |
|20110101    |20110108   |20110113  |
|20110102    |20110109   |20110114  |
|20110102    |20110109   |20110114  |
|20110102    |20110109   |20110114  |
|20110102    |20110109   |20110114  |
+------------+-----------+----------+
only showing top 20 rows


In [0]:
crm_sales_details_df.filter("sls_sales < 0").show()

crm_sales_details_df.filter("sls_quantity <= 0").show()

crm_sales_details_df.filter("sls_price < 0").show()

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO61570|    CA-1098|      17809|    20130705|   20130712|  20130717|      -18|           1|        9|
|    SO69066|  SJ-0194-L|      17923|    20131024|   20131031|  20131105|      -54|           1|       54|
|    SO69215|    TI-M823|      16864|       32154|   20131102|  20131107|      -35|           1|       35|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------

In [0]:
# Validate sales data by identifying records where sales amount does not match quantity multiplied by price.
crm_sales_details_df.filter(
    col("sls_sales") != col("sls_quantity") * col("sls_price")
)

DataFrame[sls_ord_num: string, sls_prd_key: string, sls_cust_id: int, sls_order_dt: int, sls_ship_dt: int, sls_due_dt: int, sls_sales: int, sls_quantity: int, sls_price: int]

In [0]:
from pyspark.sql.functions import count

crm_sales_details_df.groupBy("sls_ord_num") \
                    .count() \
                    .filter("count > 1") \
                    

DataFrame[sls_ord_num: string, count: bigint]

In [0]:
# Handle missing sales values by deriving them from quantity and price, and calculate missing prices from sales and quantity.
crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_sales",
    F.when(
        F.col("sls_sales").isNull(),
        F.col("sls_quantity") * F.col("sls_price")
    ).otherwise(F.col("sls_sales"))
)

crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_price",
    F.when(
        F.col("sls_price").isNull(),
        F.col("sls_sales") / F.col("sls_quantity")
    ).otherwise(F.col("sls_price"))
)

In [0]:
# Standardizing data columns
crm_sales_details_df = (
    crm_sales_details_df
    .withColumn(
        "sls_order_dt",
        F.when(
            F.col("sls_order_dt").cast("string").rlike("^[0-9]{8}$"),
            F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            F.col("sls_ship_dt").cast("string").rlike("^[0-9]{8}$"),
            F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            F.col("sls_due_dt").cast("string").rlike("^[0-9]{8}$"),
            F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd")
        )
    )
)

In [0]:
# 
# Ensure sales prices are stored as positive values.
crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_price",
    F.abs(F.col("sls_price"))
)

In [0]:
# same for sales
crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_sales",
    F.abs(F.col("sls_sales"))
)

In [0]:
# Filling missing sales prices by calculating them from sales amount and quantity.
crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_price",
    F.when(
        F.col("sls_price").isNull(),
        F.col("sls_sales") / F.col("sls_quantity")
    ).otherwise(F.col("sls_price"))
)

In [0]:
crm_sales_details_df = crm_sales_details_df.withColumn(
    "sls_sales",
    F.col("sls_quantity") * F.col("sls_price")
)

In [0]:
crm_sales_details_df = crm_sales_details_df.dropDuplicates()

In [0]:
crm_sales_details_df = crm_sales_details_df.filter(
    F.col("sls_quantity") > 0
)

In [0]:
crm_sales_details_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_sales_details")

# Gold layer (Making dimension tables and fact table and establishing relationships using surrogate keys)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

1. First dimension table: dim_customer ->using the below 3 tables which contains information about customers.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cust_info = spark.table("silver.crm_cust_info")
cust_extra = spark.table("silver.erp_cust_az12")
cust_location = spark.table("silver.erp_loc_a101")

Data validation

In [0]:
cust_info.select("cst_id","cst_key").show(10, False)

+------+----------+
|cst_id|cst_key   |
+------+----------+
|11000 |AW00011000|
|11001 |AW00011001|
|11002 |AW00011002|
|11003 |AW00011003|
|11004 |AW00011004|
|11005 |AW00011005|
|11006 |AW00011006|
|11007 |AW00011007|
|11008 |AW00011008|
|11009 |AW00011009|
+------+----------+
only showing top 10 rows


Tranform data

In [0]:
# Prepare ERP customer and location data by creating standardized customer keys.
cust_extra = (
    spark.table("silver.erp_cust_az12")
    .withColumn(
        "customer_key",
        F.regexp_replace(F.col("CID"), "^NAS", "")
    )
)

cust_location = (
    spark.table("silver.erp_loc_a101")
    .withColumn(
        "customer_key",
        F.regexp_replace(F.col("CID"), "-", "")
    )
)

In [0]:
# Build the customer dimension by integrating CRM and ERP customer information.
dim_customer = (
    cust_info.alias("c")
    .join(
        cust_extra.alias("e"),
        F.col("c.cst_key") == F.col("e.customer_key"),
        "left"
    )
    .join(
        cust_location.alias("l"),
        F.col("c.cst_key") == F.col("l.customer_key"),
        "left"
    )
)

In [0]:
# Select and rename customer dimension attributes for the Gold layer.
dim_customer = dim_customer.select(
    F.col("c.cst_id").alias("customer_id"),
    F.col("c.cst_key").alias("customer_key"),
    F.col("c.cst_firstname").alias("first_name"),
    F.col("c.cst_lastname").alias("last_name"),
    F.col("c.cst_marital_status").alias("marital_status"),
    F.col("c.cst_gndr").alias("gender"),
    F.col("e.BDATE").alias("birth_date"),
    F.col("l.CNTRY").alias("country")
)

In [0]:
# Add a unique surrogate key to the customer dimension.
from pyspark.sql.window import Window

window = Window.orderBy("customer_id")

dim_customer = dim_customer.withColumn(
    "customer_sk",
    F.row_number().over(window)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Organize and finalize the customer dimension structure with required attributes.
dim_customer = dim_customer.select(
    "customer_sk",
    "customer_id",
    "customer_key",
    "first_name",
    "last_name",
    "marital_status",
    "gender",
    "birth_date",
    "country"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# save as delta table in the Gold layer
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_customer")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Second Dimension table: dim_product-> using the below 2 tables which contain information about products

In [0]:
prd_info = spark.table("silver.crm_prd_info")
prd_cat = spark.table("silver.erp_px_cat_g1v2")

In [0]:
# Create a standardized category ID from the product key.
prd_info = prd_info.withColumn(
    "category_id",
    F.regexp_replace(
        F.concat_ws(
            "-",
            F.split(F.col("prd_key"), "-")[0],
            F.split(F.col("prd_key"), "-")[1]
        ),
        "-",
        "_"
    )
)

In [0]:
# Extract and create the product code from the product key components.
prd_info = prd_info.withColumn(
    "product_code",
    F.concat_ws(
        "-",
        F.element_at(F.split("prd_key", "-"), -3),
        F.element_at(F.split("prd_key", "-"), -2),
        F.element_at(F.split("prd_key", "-"), -1)
    )
)

In [0]:
# Join product information with product category details using category_id.
dim_product = (
    prd_info.alias("p")
    .join(
        prd_cat.alias("c"),
        F.col("p.category_id") == F.col("c.ID"),
        "left"
    )
)

In [0]:
# Generate a unique surrogate key for each product record using prd_id ordering.
window = Window.orderBy("prd_id")

dim_product = dim_product.withColumn(
    "product_sk",
    F.row_number().over(window)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_product = dim_product.select(
    "product_sk",
    F.col("prd_id").alias("product_id"),
    F.col("product_code"),
    F.col("prd_key").alias("product_key"),
    F.col("prd_nm").alias("product_name"),
    F.col("prd_cost").alias("product_cost"),
    F.col("prd_line").alias("product_line"),
    F.col("CAT").alias("category"),
    F.col("SUBCAT").alias("subcategory"),
    F.col("MAINTENANCE").alias("maintenance"),
    F.col("prd_start_dt").alias("start_date"),
    F.col("prd_end_dt").alias("end_date")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_product")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Third table: Fact table-> using the sales detail and two dimension tables

In [0]:
sales = spark.table("silver.crm_sales_details")
dim_customer = spark.table("gold.dim_customer")
dim_product = spark.table("gold.dim_product")

In [0]:
# Join sales data with customer and product dimensions to create the fact sales dataset.
fact_sales = (
    sales.alias("s")
    .join(
        dim_customer.alias("c"),
        F.col("s.sls_cust_id") == F.col("c.customer_id"),
        "left"
    )
    .join(
        dim_product.alias("p"),
        F.col("s.sls_prd_key") == F.col("p.product_code"),
        "left"
    )
)

In [0]:
# Select and rename required columns to define the final fact sales table structure.
fact_sales = fact_sales.select(
    F.col("s.sls_ord_num").alias("order_number"),
    F.col("c.customer_sk"),
    F.col("p.product_sk"),
    F.col("s.sls_order_dt").alias("order_date"),
    F.col("s.sls_ship_dt").alias("ship_date"),
    F.col("s.sls_due_dt").alias("due_date"),
    F.col("s.sls_quantity").alias("quantity"),
    F.col("s.sls_price").alias("price"),
    F.col("s.sls_sales").alias("sales")
)

In [0]:
# Validating
fact_sales.printSchema()

root
 |-- order_number: string (nullable = true)
 |-- customer_sk: integer (nullable = true)
 |-- product_sk: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- due_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales: double (nullable = true)



In [0]:
# saving as delta table 
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_sales")

# Conclusion
-Architecture: Bronze → Silver → Gold (Delta Lake)

-Source Systems: CRM & ERP (6 tables)

-Gold Tables: dim_customer, dim_product, fact_sales

-Data Quality: Missing values, duplicates, invalid dates, categorical standardization

-Modeling: Star Schema with surrogate keys

-Outcome: Analytics-ready sales warehouse